<a href="https://colab.research.google.com/github/dakshsaini77/smartkart-churn-prediction/blob/main/SmartKart_Churn_Prediction_ML_Pipeline_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 SmartKart — Customer Churn Prediction
## Complete No-Code-Concept, Full-Code ML Pipeline (Logistic Regression)

**Course:** Introduction to AI & ML | BBA AI/ML | Chitkara Business School
**CLO:** CLO02 — Apply data preprocessing, feature selection & ML models to business scenarios; evaluate performance using appropriate metrics
**Dataset:** `SmartKart_dirty_100_rows.csv` (100 customers, intentionally messy — duplicates, missing values, invalid entries, outliers)
**Business Goal:** Predict which SmartKart customers are likely to **churn** (leave), so the retention team can act before they do.

### Pipeline followed (15 steps)
1. Data Collection → 2. Data Understanding → 3. Data Cleaning → 4. Outlier Detection & Treatment → 5. Feature Selection → 6. Define Target Variable → 7. Encode Target Variable → 8. Train-Test Split → 9. Feature Standardisation → 10. Model Building → 11. Model Training → 12. Prediction → 13. Model Evaluation → 14. Model Interpretation → 15. Final Output

> **How to use this notebook:** Run every code cell top to bottom (Runtime → Run all). Each code cell is fully commented. Each markdown cell above a code cell explains *what* and *why*; each markdown cell below a code cell explains *how to read the output*.


---
## Step 1 — Data Collection

**What:** Get the raw dataset into the notebook.
**Why:** Every ML pipeline starts with raw data — here it's a CSV export of SmartKart's customer database (already collected by the business; our job starts at reading it in).

Run the cell below, then click **"Choose Files"** in the widget that appears and upload `SmartKart_dirty_100_rows.csv` from your computer.


In [ ]:
# ============================================================
# STEP 1: DATA COLLECTION
# ============================================================

# Import core libraries we'll use across the whole pipeline
import pandas as pd                      # for data handling (tables)
import numpy as np                       # for numerical operations
import matplotlib.pyplot as plt          # for plots
import seaborn as sns                    # for nicer plots (confusion matrix heatmap)

# Import Colab's file-upload widget
from google.colab import files

# This opens a "Choose Files" button — upload SmartKart_dirty_100_rows.csv here
uploaded = files.upload()

# Read the uploaded CSV into a pandas DataFrame (a table, like an Excel sheet)
df = pd.read_csv('SmartKart_dirty_100_rows.csv')

# Show the first 5 rows to confirm it loaded correctly
df.head()


---
## Step 2 — Data Understanding / Data Inspection

**What:** Look at the shape, column types, and quality of the data *before* touching anything.
**Why:** You can't clean what you haven't inspected. This step tells us exactly which problems exist so Step 3 can fix them deliberately, not by guessing.


In [ ]:
# ============================================================
# STEP 2: DATA UNDERSTANDING / DATA INSPECTION
# ============================================================

# How many rows and columns do we have?
print("Shape (rows, columns):", df.shape)

# Column names and their data types
print("\nColumn data types:")
print(df.dtypes)

# Are any columns storing numbers as text (object)? This is a common dirty-data sign.
print("\nFull info:")
df.info()

# How many missing (blank) values does each column have?
print("\nMissing values per column:")
print(df.isna().sum())

# How many fully duplicated rows exist (every column identical)?
print("\nFully duplicated rows:", df.duplicated().sum())

# Quick statistical summary — look for impossible values (negative spend, huge outliers)
print("\nStatistical summary:")
df.describe(include='all')


**Interpretation (what you should see):**
- **Shape:** 100 rows × 5 columns (`Customer_ID`, `Age`, `Monthly_Spend`, `Complaints`, `Churn`).
- **`Age` is stored as an object (text), not a number** — that's a red flag. It's because a few entries contain whitespace (`" 25 "`) or words (`"thirty"`), which forces pandas to treat the whole column as text.
- **Missing values:** a few blanks in `Age`, `Monthly_Spend`, and `Complaints`.
- **Duplicated rows: 5** — the same customer record appears twice (e.g., customer `C005` appears at row 4 and again at row 95).
- **`describe()`** reveals a `Monthly_Spend` minimum of **-1000** (spending can't be negative — invalid) and a maximum of **99999** (extreme outlier), plus a `Complaints` maximum of **50** (unrealistic for a retail customer). These get fixed in Steps 3–4.


---
## Step 3 — Data Cleaning

**What:** Fix the concrete problems found in Step 2:
- Remove duplicate records
- Remove unnecessary whitespace (`" C004 "` → `"C004"`)
- Correct data types (`Age` text → numeric; `"thirty"` → `30`)
- Remove invalid values (negative `Monthly_Spend`, impossible `Age` like `-5` or `150`)
- Handle missing values (fill with the **median**, which is safer than the mean when outliers are present)

**Why:** A model trained on dirty data learns the wrong patterns — "garbage in, garbage out." Cleaning is usually 60–70% of real-world ML work.


In [ ]:
# ============================================================
# STEP 3: DATA CLEANING
# ============================================================

print("Rows before cleaning:", len(df))

# --- 3a. Remove unnecessary whitespace from text columns ---
df['Customer_ID'] = df['Customer_ID'].astype(str).str.strip()   # " C004 " -> "C004"
df['Age'] = df['Age'].astype(str).str.strip()                   # " 25 "   -> "25"

# --- 3b. Remove duplicate records (identical rows = same customer counted twice) ---
before = len(df)
df = df.drop_duplicates()
print("Duplicate rows removed:", before - len(df))

# --- 3c. Correct data types: fix text numbers before converting Age to numeric ---
df['Age'] = df['Age'].replace({'thirty': '30'})     # fix the one spelled-out number
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')  # anything still non-numeric becomes NaN

# --- 3d. Remove invalid values ---
# Age must realistically be between 15 and 90 for a retail customer
df.loc[(df['Age'] < 15) | (df['Age'] > 90), 'Age'] = np.nan

# Monthly_Spend cannot be negative (you can't spend -1000 rupees)
invalid_spend_count = (df['Monthly_Spend'] < 0).sum()
print("Invalid negative Monthly_Spend values found:", invalid_spend_count)
df.loc[df['Monthly_Spend'] < 0, 'Monthly_Spend'] = np.nan

# --- 3e. Handle missing values: fill with median (robust to outliers, unlike mean) ---
for col in ['Age', 'Monthly_Spend', 'Complaints']:
    median_value = df[col].median()
    missing_count = df[col].isna().sum()
    df[col] = df[col].fillna(median_value)
    print(f"{col}: filled {missing_count} missing values with median = {median_value}")

print("\nRows after cleaning:", len(df))
print("\nRemaining missing values (should all be 0):")
print(df.isna().sum())


**Interpretation (what you should see):**
- **5 duplicate rows removed** → dataset shrinks from 100 to 95 rows.
- **1 invalid negative `Monthly_Spend` value found** (the `-1000` entry) — converted to missing, then filled.
- After filling, **`Age`** has its invalid entries (whitespace-only, `"thirty"`, `-5`, `150`) replaced with the column median (**40**).
- **`Monthly_Spend`** and **`Complaints`** missing values are also filled with their medians.
- **All missing-value counts should now read 0** — the dataset is clean and every column is the correct numeric type.


---
## Step 4 — Outlier Detection & Treatment

**What:** Even after cleaning, some *valid-looking* values are extreme outliers (e.g., `Monthly_Spend = 99999`, `Complaints = 50`). We detect these using the **IQR (Interquartile Range) method** and **cap** them instead of deleting the row (we don't want to lose a real customer's other information).
**Why:** Logistic Regression is sensitive to extreme values — one outlier can distort the whole model's decision boundary.


In [ ]:
# ============================================================
# STEP 4: OUTLIER DETECTION & TREATMENT (IQR method)
# ============================================================

def treat_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)   # 25th percentile
    Q3 = dataframe[column].quantile(0.75)   # 75th percentile
    IQR = Q3 - Q1                            # interquartile range
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = ((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum()
    print(f"{column}: {outlier_count} outlier(s) found | valid range = [{lower_bound:.1f}, {upper_bound:.1f}]")

    # Cap (winsorize) instead of deleting — keeps the customer's other data usable
    dataframe[column] = dataframe[column].clip(lower_bound, upper_bound)
    return dataframe

df = treat_outliers_iqr(df, 'Monthly_Spend')
df = treat_outliers_iqr(df, 'Complaints')

print("\nStatistical summary after outlier treatment:")
df[['Age', 'Monthly_Spend', 'Complaints']].describe()


**Interpretation (what you should see):**
- **`Monthly_Spend`: 1 outlier found** — the `99999` entry gets capped down to roughly **₹13,650** (the upper valid bound), so it stops dominating the model.
- **`Complaints`: 1 outlier found** — the `50` entry gets capped down to roughly **8–9 complaints**, a far more realistic maximum.
- The new `describe()` table should show a sensible maximum for both columns now — no more four/five-digit jumps.


---
## Step 5 — Feature Selection

**What:** Decide which columns are actually useful for *predicting churn* (features/inputs) and which are not.
**Why:** `Customer_ID` is just a label (like a name) — it has no predictive relationship with churn and would only confuse the model. `Age`, `Monthly_Spend`, and `Complaints` are business-meaningful drivers of customer behaviour, so we keep them.


In [ ]:
# ============================================================
# STEP 5: FEATURE SELECTION
# ============================================================

# Features (inputs) the model will learn from
selected_features = ['Age', 'Monthly_Spend', 'Complaints']

# Customer_ID is an identifier, not a predictive feature — we exclude it from X
X = df[selected_features]

print("Selected features for modeling:", selected_features)
print("\nFeature preview:")
X.head()


**Interpretation:** `X` now contains only the 3 business-meaningful numeric columns. `Customer_ID` still exists in `df` (useful later in Step 15 to identify *which* customer is at risk) but is deliberately excluded from the model's inputs.


---
## Step 6 — Define Target Variable

**What:** The target variable is what we are trying to *predict* — here, `Churn` (1 = customer left, 0 = customer stayed).
**Why:** Every supervised learning problem needs a clearly defined target (`y`) that the model learns to map the features (`X`) to.


In [ ]:
# ============================================================
# STEP 6: DEFINE TARGET VARIABLE
# ============================================================

y = df['Churn']

print("Target variable: Churn")
print("\nClass distribution:")
print(y.value_counts())
print("\nClass distribution (%):")
print((y.value_counts(normalize=True) * 100).round(1))


**Interpretation (what you should see):** Roughly **51 customers churned (Churn=1)** and **44 did not (Churn=0)** — about a **54% / 46%** split. This is a fairly *balanced* target, which is good news: the model won't be biased toward predicting only the majority class.


---
## Step 7 — Encode Target Variable

**What:** Convert the target into a numeric format the model can use.
**Why:** Logistic Regression needs numbers, not text categories like "Yes"/"No". Here `Churn` is **already numeric (0/1)**, so no conversion is needed — but we verify this explicitly rather than assuming it, since a real dataset might have "Yes"/"No" instead.


In [ ]:
# ============================================================
# STEP 7: ENCODE TARGET VARIABLE (verification step)
# ============================================================

print("Unique values in Churn:", y.unique())
print("Data type of Churn:", y.dtype)

# If Churn were text (e.g., "Yes"/"No"), we would encode it like this:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y = le.fit_transform(y)   # "No" -> 0, "Yes" -> 1

print("\nNo encoding needed — Churn is already numeric (0 = No churn, 1 = Churn).")


**Interpretation:** `Churn` contains only `[1, 0]` and is already an integer type — confirmed ready for modeling with zero extra transformation.


---
## Step 8 — Train-Test Split

**What:** Split the data into a **training set** (model learns from this) and a **testing set** (model is *evaluated* on this, unseen data).
**Why:** If we test the model on the same data it trained on, it can "memorize" answers and look artificially accurate. An 80/20 split with **stratification** (keeping the churn ratio consistent in both sets) gives a fair, honest evaluation.


In [ ]:
# ============================================================
# STEP 8: TRAIN-TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20% of data reserved for testing
    random_state=42,      # fixes the randomness so results are reproducible
    stratify=y            # keeps the same churn/no-churn ratio in both train and test sets
)

print("Training set size:", X_train.shape[0], "customers")
print("Testing set size:", X_test.shape[0], "customers")
print("\nChurn ratio in training set:")
print(y_train.value_counts(normalize=True).round(2))
print("\nChurn ratio in testing set:")
print(y_test.value_counts(normalize=True).round(2))


**Interpretation (what you should see):** **76 customers** for training and **19 customers** held out for testing. Both sets should show a similar churn ratio (roughly 54% churned / 46% not) thanks to `stratify=y` — this means the test set is a fair, representative sample.


---
## Step 9 — Feature Standardisation

**What:** Rescale numeric features so they're on a comparable scale (mean = 0, standard deviation = 1).
**Why:** `Monthly_Spend` ranges in the thousands while `Complaints` ranges 0–9. Without scaling, Logistic Regression would treat `Monthly_Spend` as "more important" purely because its numbers are bigger — standardisation fixes this. **Important:** we `fit` the scaler only on training data, then `transform` the test data with those same stats, to avoid leaking test-set information into training.


In [ ]:
# ============================================================
# STEP 9: FEATURE STANDARDISATION
# ============================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn the mean/std from the TRAINING data only...
X_train_scaled = scaler.fit_transform(X_train)

# ...then apply that same scaling to the TEST data (never re-fit on test data)
X_test_scaled = scaler.transform(X_test)

print("Before scaling (first row):", X_train.iloc[0].values)
print("After scaling  (first row):", X_train_scaled[0])


**Interpretation:** The "after scaling" values are now centered around 0 (typically between -2 and +2), regardless of whether the original feature was `Age` (18–60) or `Monthly_Spend` (₹652–₹13,650). All three features now contribute fairly to the model.


---
## Step 10 — Model Building

**What:** Instantiate the algorithm we chose for this business problem: **Logistic Regression**.
**Why:** Churn is a **binary classification problem** (Churn / No Churn) — Logistic Regression is the standard, highly-interpretable go-to algorithm for this, and it directly outputs a *probability* of churn (useful for ranking "at-risk" customers in Step 15).


In [ ]:
# ============================================================
# STEP 10: MODEL BUILDING
# ============================================================

from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model object (not yet trained)
model = LogisticRegression(random_state=42)

print("Model created:", model)


**Interpretation:** At this point the model exists but hasn't learned anything yet — it has default, untrained parameters. Training happens next.


---
## Step 11 — Model Training

**What:** Fit the model to the training data — it learns the relationship between `Age`, `Monthly_Spend`, `Complaints` and `Churn`.
**Why:** This is the actual "learning" step; the algorithm adjusts its internal weights (coefficients) to best separate churners from non-churners in the training data.


In [ ]:
# ============================================================
# STEP 11: MODEL TRAINING
# ============================================================

model.fit(X_train_scaled, y_train)

print("Model training complete.")
print("Learned coefficients:", model.coef_[0])
print("Learned intercept:", model.intercept_[0])


**Interpretation:** The model has now learned one **coefficient per feature** (how strongly and in which direction each feature pushes toward churn) plus an intercept (baseline). We'll interpret exactly what these numbers mean in Step 14.


---
## Step 12 — Prediction

**What:** Use the trained model to predict churn on the **test set** — data it has never seen before.
**Why:** This simulates how the model would perform on *new, real* SmartKart customers.


In [ ]:
# ============================================================
# STEP 12: PREDICTION
# ============================================================

# Predicted class (0 = No Churn, 1 = Churn)
y_pred = model.predict(X_test_scaled)

# Predicted probability of churn (more useful for business ranking — see Step 15)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Predicted classes for test customers:", y_pred)
print("\nPredicted churn probabilities (first 5):", y_pred_proba[:5].round(2))


**Interpretation:** `y_pred` gives a hard 0/1 decision per test customer; `y_pred_proba` gives the underlying confidence (e.g., `0.82` means the model is 82% sure this customer will churn). We keep both — the probability is what actually powers a real retention dashboard.


---
## Step 13 — Model Evaluation

**What:** Measure how good the predictions actually are, using the standard classification metrics from the CHO: **Confusion Matrix, Accuracy, Precision, Recall, F1-Score**.
**Why:** "It works" isn't good enough for a business decision — we need to know exactly how often the model is right, and in *which direction* it makes mistakes (missing real churners is often costlier than a false alarm).


In [ ]:
# ============================================================
# STEP 13: MODEL EVALUATION
# ============================================================

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print(f"\nAccuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1-Score:  {f1:.2%}")
print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Visualise the confusion matrix as a heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: No Churn', 'Predicted: Churn'],
            yticklabels=['Actual: No Churn', 'Actual: Churn'])
plt.title('SmartKart Churn Prediction — Confusion Matrix')
plt.tight_layout()
plt.show()


**Interpretation (what you should see, approximately):**
- **Accuracy ≈ 89–95%** — the model correctly classifies most test customers.
- **Recall ≈ 100%** — the model catches *every actual churner* in the test set. For a churn-prevention business problem, **this matters most**: missing a real churner means losing a paying customer silently.
- **Precision ≈ 83–91%** — a small number of loyal customers get incorrectly flagged as "at risk" (false alarms). This is an acceptable trade-off: it's cheaper for the retention team to send an unnecessary discount offer than to lose a customer.
- **Confusion Matrix reading:** top-left = correctly predicted "No Churn"; bottom-right = correctly predicted "Churn"; the off-diagonal cells are the mistakes (almost all in the "false alarm" direction, not the "missed churner" direction — which is the safer type of error here).


---
## Step 14 — Model Interpretation

**What:** Look *inside* the model's coefficients to understand **which factors actually drive churn**, and in which direction.
**Why:** For a BBA/business audience, the prediction alone isn't enough — the real value is telling management "*this* is why customers leave" so they can act on the root cause, not just the symptom.


In [ ]:
# ============================================================
# STEP 14: MODEL INTERPRETATION
# ============================================================

coefficients = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("Feature impact on churn (sorted by strength):")
print(coefficients)

print("\nHow to read this:")
for _, row in coefficients.iterrows():
    direction = "INCREASES" if row['Coefficient'] > 0 else "DECREASES"
    print(f"- {row['Feature']}: coefficient = {row['Coefficient']:.2f} -> {direction} churn risk as it goes up")


**Interpretation (what you should see):**
- **`Monthly_Spend` has a strong *negative* coefficient** → the more a customer spends per month, the **lower** their churn risk. High spenders are your most loyal, highest-value customers.
- **`Complaints` has a strong *positive* coefficient** → each additional complaint **increases** churn risk substantially. This is the clearest actionable insight: SmartKart's customer-support/complaint-resolution process is a direct lever on retention.
- **`Age` has a small *positive* coefficient** → a mild tendency for churn risk to rise slightly with age in this dataset, but it's the weakest of the three factors.

**Business takeaway to write in your report:** *"Reducing customer complaints and protecting high-spend customer relationships are the two most effective retention levers SmartKart has."*


---
## Step 15 — Final Output

**What:** Turn model predictions into a **business-usable output**: a ranked list of which test customers are "Likely to Churn" vs "Not Likely to Churn", with the highest-risk customers flagged first.
**Why:** This is the actual deliverable a retention/marketing team would use — not raw model metrics, but an action list.


In [ ]:
# ============================================================
# STEP 15: FINAL OUTPUT — BUSINESS-READY RESULTS
# ============================================================

# Rebuild a results table aligned with the test set's original index
results = X_test.copy()
results['Customer_ID'] = df.loc[X_test.index, 'Customer_ID'].values
results['Actual_Churn'] = y_test.values
results['Predicted_Churn'] = y_pred
results['Churn_Probability'] = y_pred_proba.round(3)

# Human-readable label for business users
results['Risk_Label'] = np.where(results['Predicted_Churn'] == 1, 'Likely to Churn', 'Not Likely to Churn')

# Sort so the highest-risk customers appear first — this is the action list for the retention team
results = results.sort_values(by='Churn_Probability', ascending=False)

# Reorder columns for a clean, presentable table
results = results[['Customer_ID', 'Age', 'Monthly_Spend', 'Complaints',
                    'Actual_Churn', 'Predicted_Churn', 'Churn_Probability', 'Risk_Label']]

print("SmartKart — Customer Churn Risk Report (Test Set)")
display(results)

# Flag the top 5 highest-risk customers for immediate retention action
print("\nTop 5 customers requiring IMMEDIATE retention attention:")
display(results.head(5))

# Save the report as a CSV — this is your deliverable to upload to GitHub
results.to_csv('smartkart_churn_risk_report.csv', index=False)
files.download('smartkart_churn_risk_report.csv')
print("\nSaved and downloaded: smartkart_churn_risk_report.csv")


**Interpretation:** `smartkart_churn_risk_report.csv` is now on your computer — a ranked, business-ready table the SmartKart retention team could act on directly (e.g., send a retention offer to everyone labeled "Likely to Churn", starting from the top of the list).

---
## ✅ Summary — What You Just Built
You took a **dirty, real-world-style dataset** through the **complete 15-step ML pipeline** — from raw CSV to a business-ready churn risk report — using **Logistic Regression**, and interpreted every result in business terms, not just code output.

### 📤 Push this to GitHub
1. In Colab: **File → Download → Download .ipynb**
2. In your `bba-ai-ml-portfolio` repo, create folder `part-a/day03-supervised-learning/`
3. Upload this notebook **and** `smartkart_churn_risk_report.csv` there
4. Commit with message: `"Day 3: SmartKart churn prediction - full ML pipeline (Logistic Regression)"`

### 📝 Resume line unlocked
> "Built an end-to-end customer churn prediction pipeline (data cleaning, outlier treatment, feature engineering, Logistic Regression) on a 100-record retail dataset, achieving ~90% accuracy and 100% recall on churn detection; delivered a business-ready risk report."
